# Multimodal Pneumonia Detection: Chest X-Ray vs. CT Scan
## With Cross-Subject (Group-wise) Validation & Best Prediction Ensembles

## Overview & Key Contributions
This project builds upon the popular Kaggle benchmark **Pneumonia Detection using CNN (92.6% Accuracy)** by Madhav Mathur (`madz2000`) and extends it into a **Multimodal Diagnostic System** supporting both **Chest X-Ray** and **Chest CT Scan** imaging:

1. **Cross-Subject (Group-Wise) K-Fold Validation**:
   - Eliminates **Subject Data Leakage** by grouping patient scans (`person{ID}` for X-Ray, `ct_patient{ID}` for CT Scans) with `StratifiedGroupKFold` so that $train\_subjects \cap val\_subjects = \emptyset$.

2. **Multimodal Modality Comparison (Chest X-Ray vs. CT Scan)**:
   - Evaluates 5-fold cross-subject models for both imaging modalities.
   - Analyzes clinical trade-offs: CT Scans provide high 3D spatial resolution for early Ground-Glass Opacities (GGO), whereas Chest X-Rays serve as low-cost, low-radiation screening.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from dataset_loader import PneumoniaDatasetLoader
from cnn_model import get_model
from cross_subject_cv import CrossSubjectValidator
from best_prediction import BestPredictionEngine

print("Multimodal libraries initialized!")

## 1. Multimodal Dataset Loading & Patient ID Extraction
Loading Chest X-Ray and Chest CT Scan datasets with patient subject grouping.

In [ ]:
data_dir = './data'
loader = PneumoniaDatasetLoader(img_size=(150, 150))

# Load X-Ray & CT Scan datasets
xray_records = loader.load_dataset_index(data_dir, modality='xray')
if len(xray_records) < 20:
    xray_records = loader.generate_synthetic_dataset(data_dir, modality='xray', num_subjects=60, images_per_subject=6)

ct_records = loader.load_dataset_index(data_dir, modality='ct_scan')
if len(ct_records) < 20:
    ct_records = loader.generate_synthetic_dataset(data_dir, modality='ct_scan', num_subjects=60, images_per_subject=6)

print(f"Chest X-Ray Dataset: {len(xray_records)} images across {len(set([r['subject_id'] for r in xray_records]))} subjects")
print(f"Chest CT Scan Dataset: {len(ct_records)} images across {len(set([r['subject_id'] for r in ct_records]))} subjects")

## 2. 5-Fold Cross-Subject Validation (X-Ray & CT Scan)

In [ ]:
# Run Cross-Subject CV for X-Ray
xray_validator = CrossSubjectValidator(xray_records, modality='xray', n_splits=5, checkpoint_dir='./checkpoints')
xray_summary, xray_oof_probs, xray_oof_targets = xray_validator.run_cross_validation(epochs=4, lr=0.0005, batch_size=16)

# Run Cross-Subject CV for CT Scan
ct_validator = CrossSubjectValidator(ct_records, modality='ct_scan', n_splits=5, checkpoint_dir='./checkpoints')
ct_summary, ct_oof_probs, ct_oof_targets = ct_validator.run_cross_validation(epochs=4, lr=0.0005, batch_size=16)

## 3. Side-by-Side Modality Performance Comparison
Comparing Chest X-Ray vs CT Scan performance metrics.

In [ ]:
comparison = BestPredictionEngine.generate_modality_comparison(xray_summary, ct_summary)
print(json.dumps(comparison, indent=2))